# Generating tokens for CityScapes class set
this notebook has been adapted from https://github.com/baaivision/tokenize-anything/blob/main/notebooks/concept.ipynb to add weights for CityScapes 

## Overview

The semantic token is pre-trained to align EVACLIP-5B on SA-1B masks. This guide shows you how to make the concept weights.

## Setup

Necessary imports, models and functions for making.

In [1]:
import pickle
import sys
sys.path.append("../../EVA/EVA-CLIP/rei")

import torch
from eva_clip import create_model_and_transforms, get_tokenizer

model_name = 'EVA02-CLIP-bigE-14-plus'
# Download from https://github.com/baaivision/EVA/tree/master/EVA-CLIP
pretrained = '../weights/EVA02_CLIP_E_psz14_plus_s9B.pt'

inference_mode = torch.inference_mode()
inference_mode.__enter__()

def generate_concept_weights(model, tokenizer, device, concepts, templates):
    concept_embeds = []
    for concept in concepts:
        texts = [template.format(concept) for template in templates]
        texts = tokenizer(texts).to(device=device)
        embeds = model.encode_text(texts).float()
        embeds = torch.nn.functional.normalize(embeds, dim=-1)
        if len(templates) > 1:
            embed = embeds.mean(dim=0)
            embed = torch.nn.functional.normalize(embed, dim=-1)
        else:
            embed = embeds[0]
        concept_embeds.append(embed)
    return torch.stack(concept_embeds, dim=-1)


Please 'pip install apex'


## Concepts

Following concepts are used to pre-train TAP.

In [2]:
CITYSCAPES_CLASS_NAMES = [
    "road",
    "sidewalk",
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "on rails",
    "motorcycle",
    "bicycle",
    "building",
    "wall",
    "fence",
    "pole",
    "traffic sign",
    "traffic light",
    "vegetation",
    "terrain",
    "sky"
]

In [3]:
concepts = (
    CITYSCAPES_CLASS_NAMES
)
concepts = set([name.lower() for name in concepts])
remove = set()
for singular in concepts:
    for plural in [singular + "s", singular + "es"]:
        if plural in concepts:
            remove.add(plural)
concepts = sorted(list(concepts.difference(remove)))
print(len(concepts), "concepts.")


19 concepts.


## Build CLIP


In [4]:
import os
device = "cuda" # if torch.cuda.is_available() else "cpu"
assert os.path.exists(pretrained), f"Pretrained model not found at {pretrained}"

model, _, preprocess = create_model_and_transforms(model_name, pretrained, force_custom_clip=True)
tokenizer = get_tokenizer(model_name)
model = model.to(device)


## Make Concept Weights

In [6]:

concept_weights = generate_concept_weights(model, tokenizer, device, concepts, ['a {}'])
concept_weights = concept_weights * model.logit_scale.data.exp()
concept_weights = concept_weights.cpu().numpy()
with open('../weights/cityscapes-19.pkl', 'wb') as f:
    pickle.dump([concept_weights, concepts], f)
print(concept_weights.shape, concept_weights.dtype, (concept_weights.min(), concept_weights.max()))


(1024, 19) float32 (-14.032139, 14.868487)
